In [95]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [96]:
from dotenv import load_dotenv
import os
from groq import Groq
import re
from pathlib import Path

load_dotenv()

TASKS_FILE = Path.cwd() / "tasks.txt"

model = 'llama-3.3-70b-versatile'

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)


Classe Agent, uma abstração do agente. Na rotina de iniciar define algumas propriedades e caso seja passado, insere o system prompt inicial que instrui o modelo a agir. 

Quando a classe é chamada, caso passe uma mensagem é uma entrada do usuário, se não é uma entrada do assistente. 

Implementa a função de execução que realiza as requisições para o modelo. 

In [97]:
class Agent:
    def __init__(self, client: Groq, system: str = "") -> None:
        self.client = client
        self.system = system
        self.messages: list = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message=""):
        print(' > ', message)
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        print(' | ', result)
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model=model, messages=self.messages
        )
        return completion.choices[0].message.content

System prompt base que instrui como o agente deve agir. 
Importante ressaltar que é onde são estabelecidos os padrões e formatos de entradas e saídas, como o fluxo de Question > Thought, Action, Observation and Answer. 

Abaixo consta também as funções personalizadas que o agente pode acionar. 

## System prompt e funções personalizadas

In [98]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

add_task:
e.g. add_task: Buy milk
Appends one task to tasks.txt as a single line "- [ ] Buy milk" and returns OK.

list_tasks:
e.g. list_tasks: ' '
Returns all lines from tasks.txt (or "No tasks found.").


Example session:

Question: Adicione a tarefa de colocar as roupas para lavar
Thought: A tarefa a ser adicionada é colocar as roupas para lavar.
Action: add_task: Colocar as roupas para lavar
PAUSE 

You will be called again with this:

Observation: OK task saved

If you have finished, output it as the Answer.

Answer: Tarefa adicionada! 


Question: O que tenho para fazer?
Thought: A pergunta é sobre quais tarefas tenho para realizar. Para responder a essa pergunta, preciso verificar as tarefas salvas no arquivo tasks.txt.
Action: list_tasks: ' '
PAUSE  

You will be called again with this: 

Observation: '
['- [ ] outra tarefa',
 '- [ ] Colocar as roupas para lavar',]

If you have finished, output it as the Answer.

Answer: Você tem as seguintes tarefas pendentes: 
- outra tarefa
- Colocar as roupas para lavar 


Now it's your turn:

""".strip()


In [99]:
def add_task(description: str) -> str:

    if not os.path.exists(TASKS_FILE):
        open(TASKS_FILE, "w", encoding="utf-8").close()
    
    desc = (description or "").strip()

    line = f"- [ ] {desc}"

    with TASKS_FILE.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

    return f"OK task saved"




def list_tasks(_: str = "") -> str:
    TASKS_FILE.parent.mkdir(parents=True, exist_ok=True)

    if TASKS_FILE.exists():
        existing = TASKS_FILE.read_text(encoding="utf-8").splitlines()
        return existing



In [100]:
list_tasks()

['- [ ] Terminar o código da aula prática',
 '- [ ] Fazer o relatório',
 '- [ ] Lavar as vasilhas']

Afim de automatizar o fluxo de chamada de funções, o método loop() utiliza um laço de repetição 

In [101]:

def loop(question: str, max_iterations = 20):

    agent = Agent(client=client, system=system_prompt)

    tools = ["add_task", "list_tasks"]

    next_prompt = question

    i = 0

    while i < max_iterations:
        i += 1
        
        result = agent(next_prompt)        

        if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]
            

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            continue

        if "Answer" in result:
            break



In [102]:
loop(question="Tem a tarefa de lavar as vasilhas")

 >  Tem a tarefa de lavar as vasilhas
 |  Thought: A tarefa a ser adicionada é lavar as vasilhas.
Action: add_task: Lavar as vasilhas
PAUSE
 >  Observation: OK task saved
 |  Answer: Tarefa adicionada!


In [103]:
loop(question="Preciso terminar o código da aula prática e fazer o relatorio")

 >  Preciso terminar o código da aula prática e fazer o relatorio
 |  Thought: A tarefa a ser adicionada é terminar o código da aula prática e fazer o relatório.
Action: add_task: Terminar o código da aula prática e fazer o relatório
PAUSE
 >  Observation: OK task saved
 |  Answer: Tarefa adicionada! Agora você tem uma tarefa para terminar o código da aula prática e fazer o relatório registrada em sua lista de tarefas.


In [104]:
loop(question="O que tenho para fazer?")

 >  O que tenho para fazer?
 |  Thought: A pergunta é sobre quais tarefas tenho para realizar. Para responder a essa pergunta, preciso verificar as tarefas salvas no arquivo tasks.txt.
Action: list_tasks: ' '
PAUSE
 >  Observation: ['- [ ] Terminar o código da aula prática', '- [ ] Fazer o relatório', '- [ ] Lavar as vasilhas', '- [ ] Lavar as vasilhas', '- [ ] Terminar o código da aula prática e fazer o relatório']
 |  Thought: Com base na observação, é possível ver que há várias tarefas pendentes. No entanto, é importante notar que há tarefas repetidas, o que pode ser um problema. Para gerar a resposta, vou remover as duplicatas e formatar as tarefas de forma agradável.

Action: None
A resposta está pronta.

Answer: Você tem as seguintes tarefas pendentes: 
- Terminar o código da aula prática
- Fazer o relatório
- Lavar as vasilhas
- Terminar o código da aula prática e fazer o relatório
